# Sinh audio bang model da finetune

**Upload len `MyDrive/f5tts/` truoc khi chay:**

| File | Nguon | Kich thuoc |
|---|---|---|
| `ckpts/model_final.pt` | notebook train da copy sang | 1.35 GB |
| `ckpts/pretrained_vn1000h.pt` | `f5tts/F5-TTS-Vietnamese/ckpts/your_training_dataset/` | 1.3 GB |
| `colab.patch` | `f5tts/` | 3 KB |
| `tts_text.py` | `f5tts/` | 4 KB |
| `vi_norm.py` | `pipeline/` | 14 KB |
| `vocab.txt` | `f5tts/F5-TTS-Vietnamese/data/your_training_dataset/` | 30 KB |
| `sample_00004.wav` | `.../data/your_training_dataset/wavs/` | 190 KB |
| `nu.wav` | `f5tts/refs/` | 445 KB |
| `nu.txt` | `f5tts/refs/` | 214 B |

`tts_text.py` giu quy tac tach doan va chuan hoa dau cau, `vi_norm.py` doc so thanh chu — **cung file ma `monitor_server.py` tren Mac dung**, nen audio sinh o hai noi giong nhau. Sua quy tac thi upload lai file nay.

Giong **nam** dung model da finetune (`model_final.pt`), giong **nu** dung model pretrained goc (`pretrained_vn1000h.pt`) vi chua co du lieu train cho giong nu.

Chon **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/f5tts'
!ls -lh {DRIVE} {DRIVE}/ckpts

In [ ]:
%cd /content
!rm -rf F5-TTS-Vietnamese
!git clone -q https://github.com/nguyenthienhy/F5-TTS-Vietnamese
%cd /content/F5-TTS-Vietnamese
!git checkout -q e74db9d && git apply {DRIVE}/colab.patch
!pip install -q -e . --no-deps
!pip install -q accelerate cached_path click datasets ema_pytorch hydra-core jieba \
    librosa pydub pypinyin safetensors soundfile tomli torchdiffeq transformers \
    vocos x_transformers

In [ ]:
!mkdir -p /content/tts
!cp {DRIVE}/tts_text.py {DRIVE}/vi_norm.py /content/
!cp {DRIVE}/vocab.txt {DRIVE}/sample_00004.wav /content/tts/
!cp {DRIVE}/nu.wav {DRIVE}/nu.txt /content/tts/
!cp {DRIVE}/ckpts/model_final.pt {DRIVE}/ckpts/pretrained_vn1000h.pt /content/tts/
!ls -lh /content/tts

Nap model **mot lan** cho ca phien. Chay lai cac cell sinh audio ben duoi bao nhieu lan cung duoc ma khong phai nap lai.

In [ ]:
import sys, os
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/F5-TTS-Vietnamese/src')

from importlib.resources import files
from omegaconf import OmegaConf
from f5_tts.infer.utils_infer import (
    cfg_strength, cross_fade_duration, infer_process, load_model, load_vocoder,
    nfe_step, preprocess_ref_audio_text, sway_sampling_coef, target_rms,
)
from f5_tts.model import DiT, UNetT  # noqa: F401

VOCAB = '/content/tts/vocab.txt'

with open('/content/tts/nu.txt', encoding='utf-8') as f:
    NU_REF_TEXT = f.read().strip()

VOICES = {
    'nam': {'ckpt': '/content/tts/model_final.pt',
            'ref_audio': '/content/tts/sample_00004.wav',
            'ref_text': 'tình cờ lạc bước vào một ngôi mộ hoang của quý phi đời trước.'},
    'nu': {'ckpt': '/content/tts/pretrained_vn1000h.pt',
           'ref_audio': '/content/tts/nu.wav',
           'ref_text': NU_REF_TEXT},
}

vocoder = load_vocoder(vocoder_name='vocos')
cfg = OmegaConf.load(str(files('f5_tts').joinpath('configs/F5TTS_Base.yaml'))).model

for voice, v in VOICES.items():
    v['model'] = load_model(globals()[cfg.backbone], cfg.arch, v['ckpt'], mel_spec_type='vocos', vocab_file=VOCAB)
    v['ref_audio'], v['ref_text'] = preprocess_ref_audio_text(v['ref_audio'], v['ref_text'])

print('san sang:', ', '.join(VOICES))

In [ ]:
import io, contextlib, time, soundfile as sf
from tqdm.auto import tqdm
from tts_text import prepare_text, split_text, max_chunk_bytes, join_wavs, speed_for, wav_to_mp3, SPEED, SENTENCE_GAP

def tts(text, voice='nam', out_name='audio', speed=SPEED, gap=SENTENCE_GAP):
    v = VOICES[voice]
    ref_audio, ref_text, model = v['ref_audio'], v['ref_text'], v['model']
    limit = max_chunk_bytes(ref_audio, ref_text)
    parts = split_text(prepare_text(text), limit)
    print(f'{len(parts)} doan, nguong {limit} byte, giong {voice}')
    paths, started = [], time.time()
    for i, part in enumerate(tqdm(parts, desc=f'giong {voice}', unit=' doan')):
        with contextlib.redirect_stdout(io.StringIO()):
            audio, sr, _ = infer_process(
                ref_audio, ref_text, part, model, vocoder, mel_spec_type='vocos',
                show_info=lambda *a, **k: None, progress=None,
                target_rms=target_rms, cross_fade_duration=cross_fade_duration,
                nfe_step=nfe_step, cfg_strength=cfg_strength,
                sway_sampling_coef=sway_sampling_coef, speed=speed_for(part, speed), fix_duration=None,
            )
        path = f'/content/tts/_part{i:03d}.wav'
        sf.write(path, audio, sr)
        paths.append(path)
    wav = f'/content/tts/{out_name}.wav'
    mp3 = f'/content/tts/{out_name}.mp3'
    join_wavs(paths, wav, gap)
    for p in paths:
        os.remove(p)
    ok, error = wav_to_mp3(wav, mp3)
    os.remove(wav)
    if not ok:
        raise RuntimeError(error)
    print(f'\nxong {len(parts)} doan trong {time.time()-started:.0f}s -> {mp3}')
    return mp3

## Sinh audio

Dan van ban vao `TEXT` roi chay. Khong can viet thuong hay bo dau `:` `"` `...` — `prepare_text` tu xu ly.

In [ ]:
from IPython.display import Audio, display

VOICE = 'nam'  # 'nam' hoac 'nu'

TEXT = """
Phương Ứng Vật liền yên tâm, không phải để bản thân thực sự học lại từ đầu một lượt là được.
Kiểu đó ba bốn tháng thời gian tuyệt đối chẳng đủ dùng, bản thân lại chẳng có được cái tài "nhìn qua là nhớ".
"""

mp3 = tts(TEXT, voice=VOICE, out_name='chuong01')
display(Audio(mp3))

## Kiem tra file mp3

Do do dai, loudness va peak cua tung file trong `/content/tts`. Muc dich la **-14 LUFS**, peak phai duoi -1.5 dBTP. Lech +-1 LUFS la binh thuong.

In [ ]:
import glob, os, subprocess
from tts_text import measure_loudness, LOUDNESS_LUFS, TRUE_PEAK_DB

def duration(path):
    out = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'csv=p=0', path], capture_output=True, text=True).stdout.strip()
    return float(out) if out else 0.0

paths = sorted(glob.glob('/content/tts/*.mp3'))
print(f'{"file":<28}{"phut":>8}{"MB":>8}{"LUFS":>9}{"dBTP":>9}  ghi chu')
for path in paths:
    stats = measure_loudness(path)
    lufs, peak = stats.get('input_i'), stats.get('input_tp')
    seconds, size = duration(path), os.path.getsize(path) / 1e6
    notes = []
    try:
        if float(lufs) == float('-inf'):
            notes.append('im lang')
        elif abs(float(lufs) - LOUDNESS_LUFS) > 1.0:
            notes.append('loudness lech')
        if float(peak) > TRUE_PEAK_DB + 0.5:
            notes.append('peak cao')
    except (TypeError, ValueError):
        notes.append('khong do duoc')
    if seconds < 1:
        notes.append('qua ngan')
    print(f'{os.path.basename(path):<28}{seconds/60:>8.1f}{size:>8.1f}'
          f'{str(lufs):>9}{str(peak):>9}  {", ".join(notes) or "ok"}')
if not paths:
    print('chua co file mp3 nao trong /content/tts')


## Tai ve may

Trinh duyet se hoi cho phep tai nhieu file — bam **Allow**. Neu co nhieu file, nen nen lai roi tai mot lan (cell duoi cung).

In [ ]:
from google.colab import files
import glob

for path in sorted(glob.glob('/content/tts/*.mp3')):
    print(path)
    files.download(path)

In [ ]:
from google.colab import files

!cd /content/tts && zip -q -r /content/audio.zip *.mp3
!ls -lh /content/audio.zip
files.download('/content/audio.zip')